In [ ]:
import os
import sys

repo_root = os.path.abspath(os.getcwd())
repo_candidates = [
    repo_root,
    os.path.join(repo_root, "OW_OVD"),
    "/kaggle/working/OW_OVD",
    "/kaggle/working",
]

repo_root = None
for candidate in repo_candidates:
    if os.path.isdir(os.path.join(candidate, "yolo_world")):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the repo root containing the yolo_world package. "
        "Please set the working directory to the repo folder first."
    )

sys.path.insert(0, repo_root)
os.environ["PYTHONPATH"] = repo_root + (
    os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)
os.chdir(repo_root)

print("Changed working directory to:", os.getcwd())
print("Added repo root to PYTHONPATH:", repo_root)

# Standalone visualization notebook for 10 different classes

This notebook is designed to be easy to rerun for paper evidence.

What it does:
- reads images from your dataset root
- picks up to 10 images from 10 different classes
- runs inference with your own config and checkpoint
- draws known boxes in yellow and unknown boxes in red

Dataset folder convention expected by this notebook:
- root/test/0/<class_name>/<image files>
- if your folder is flat, the notebook will automatically fall back to the first 10 files

In [ ]:
import os
import sys
import glob
import importlib.util

# -------------------------------------------------------------------
# 0) Dùng đúng setup cũ từng chạy được trên Kaggle
# -------------------------------------------------------------------
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository already exists. Updating...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

if not os.path.exists("third_party/mmyolo"):
    print("Cloning mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("mmyolo already exists.")

# -------------------------------------------------------------------
# 1) Cài đúng stack đã từng chạy thành công trên Kaggle
# -------------------------------------------------------------------
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers
!pip install "mmdet>=3.1.0" --no-deps
!pip install --no-build-isolation --no-deps third_party/mmyolo

# -------------------------------------------------------------------
# 2) Vá lỗi version ceiling trong mmdet nếu cần
# -------------------------------------------------------------------
spec = importlib.util.find_spec("mmdet")
if spec is not None and spec.origin is not None:
    init_file = spec.origin
    with open(init_file, "r") as f:
        content = f.read()
    if "mmcv_maximum_version = '2.2.0'" in content:
        content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
        with open(init_file, "w") as f:
            f.write(content)

# -------------------------------------------------------------------
# 3) Đảm bảo repo root nằm trên PYTHONPATH để config custom_imports = ['yolo_world'] import được
# -------------------------------------------------------------------
repo_root = os.path.abspath(os.getcwd())
if os.path.isdir(os.path.join(repo_root, "yolo_world")):
    sys.path.insert(0, repo_root)
    os.environ["PYTHONPATH"] = repo_root + (
        os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
    )
    print("Added repo root to PYTHONPATH:", repo_root)
else:
    print("WARNING: repo root does not contain yolo_world package:", repo_root)

from PIL import Image, ImageDraw, ImageFont
from mmdet.apis import init_detector, inference_detector

# -------------------------------------------------------------------
# 4) Chỉnh đường dẫn theo môi trường của bạn
# -------------------------------------------------------------------
TEST_ROOT = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/test/0"
CFG_PATH = "configs/open_world/mowod/custom/ip102_t3.py"
CKPT_PATH = "/kaggle/input/models/nta2112/ow-ovd-checkpoint-task1-2-3-4/pytorch/default/1/t3_best.pth"
OUT_DIR = "/kaggle/working/OW_OVD/paper_visuals"

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------------------------
# 5) Tự động tìm checkpoint nếu bạn đang dùng pattern *.pth
# -------------------------------------------------------------------
if "*" in CKPT_PATH:
    candidates = sorted(glob.glob(CKPT_PATH, recursive=True))
    if candidates:
        CKPT_PATH = candidates[0]
        print("Using checkpoint:", CKPT_PATH)
    else:
        raise FileNotFoundError(f"Không tìm thấy checkpoint theo pattern: {CKPT_PATH}")

if not os.path.exists(TEST_ROOT):
    raise FileNotFoundError(f"Không tìm thấy dataset root: {TEST_ROOT}")
if not os.path.exists(CFG_PATH):
    raise FileNotFoundError(f"Không tìm thấy config: {CFG_PATH}")
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {CKPT_PATH}")

print("TEST_ROOT:", TEST_ROOT)
print("CFG_PATH:", CFG_PATH)
print("CKPT_PATH:", CKPT_PATH)
print("OUT_DIR:", OUT_DIR)

In [ ]:
import os
import glob

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp")


def collect_one_image_per_class(test_root, max_classes=10):
    """
    Lấy tối đa 1 ảnh cho mỗi class khác nhau.
    Nếu folder có cấu trúc <class_name>/<file>, ta lấy 1 ảnh từ từng class.
    Nếu folder phẳng, ta lấy 10 ảnh đầu tiên.
    """
    class_dirs = [d for d in sorted(os.listdir(test_root)) if os.path.isdir(os.path.join(test_root, d))]

    picked = []
    if class_dirs:
        for class_dir in class_dirs:
            class_path = os.path.join(test_root, class_dir)
            imgs = []
            for ext in IMAGE_EXTS:
                imgs.extend(sorted(glob.glob(os.path.join(class_path, f"*{ext}"))))
            if imgs:
                picked.append(imgs[0])
                if len(picked) >= max_classes:
                    break

    if len(picked) < max_classes:
        flat_imgs = []
        for ext in IMAGE_EXTS:
            flat_imgs.extend(sorted(glob.glob(os.path.join(test_root, f"*{ext}"))))
        if flat_imgs:
            for p in flat_imgs:
                if p not in picked:
                    picked.append(p)
                if len(picked) >= max_classes:
                    break

    return picked[:max_classes]

sample_paths = collect_one_image_per_class(TEST_ROOT, max_classes=10)

if not sample_paths:
    raise FileNotFoundError(f"Không tìm thấy ảnh nào trong: {TEST_ROOT}")

print("Selected images:")
for i, p in enumerate(sample_paths, start=1):
    print(f"{i:02d} - {p}")

In [ ]:
# -------------------------------------------------------------------
# 3) Load model
# -------------------------------------------------------------------
import torch
import importlib.metadata

try:
    import mmcv
    import mmyolo
    print("mmcv:", mmcv.__version__)
    print("mmyolo:", importlib.metadata.version("mmyolo"))
except AssertionError as e:
    raise RuntimeError(
        "OpenMMLab version mismatch detected before model loading. "
        "Your current environment is using an incompatible mmcv version for the repo's mmyolo. "
        "Please reinstall the compatible mmcv build used by the old working notebook, then restart the kernel."
    ) from e

# If the cell was already executed once in this kernel, unwrap the previous patch
# first so we do not recursively wrap torch.load again and again.
if getattr(torch.load, "_owd_safe_patch", False):
    torch.load = getattr(torch.load, "_owd_original", torch.load)

_orig_torch_load = torch.load

def _safe_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

_safe_torch_load._owd_safe_patch = True
_safe_torch_load._owd_original = _orig_torch_load
torch.load = _safe_torch_load

model = init_detector(CFG_PATH, CKPT_PATH, device="cpu")

class_names = []
if hasattr(model, "dataset_meta") and isinstance(model.dataset_meta, dict):
    class_names = model.dataset_meta.get("classes", [])

UNKNOWN_ID = 80
print("Loaded model successfully.")
print("Number of class names:", len(class_names))

In [ ]:
# -------------------------------------------------------------------
# 4) Predict on 10 selected images and draw boxes
# -------------------------------------------------------------------
try:
    font = ImageFont.truetype("arial.ttf", 24)
except Exception:
    font = ImageFont.load_default()

for idx, img_path in enumerate(sample_paths, start=1):
    result = inference_detector(model, img_path)
    pred_instances = result.pred_instances

    boxes = pred_instances.bboxes.cpu().numpy()
    scores = pred_instances.scores.cpu().numpy()
    labels = pred_instances.labels.cpu().numpy().astype(int)

    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    for box, score, label_id in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [int(v) for v in box]
        is_unknown = (label_id == UNKNOWN_ID) or (label_id >= len(class_names))

        if is_unknown:
            label_name = "unknown"
            color = (255, 0, 0)
        else:
            label_name = class_names[label_id] if label_id < len(class_names) else f"class_{label_id}"
            color = (255, 215, 0)

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((max(1, x1), max(1, y1 - 28)), f"{label_name} {score:.2f}", fill=color, font=font)

    out_path = os.path.join(OUT_DIR, f"pred_{idx:02d}.png")
    img.save(out_path)
    print("Saved:", out_path)

print("Done. Outputs saved to:", OUT_DIR)